# Two-Tower Retrieval, From Scratch (NumPy)

Companion notebook for *"The 10-Millisecond Trick: How Recommenders Search Millions of Items Before You Blink"*.

This notebook trains a real two-tower retrieval model with **in-batch sampled softmax** — the workhorse of the retrieval stage at Netflix/YouTube/TikTok scale — using only NumPy (manual backprop, no PyTorch needed). You'll see:

1. How user and item towers map features to a shared embedding space
2. Why training with sampled negatives works — and its limits (the hard-negatives story from Meta's July 2026 paper)
3. How retrieval becomes a nearest-neighbor lookup over precomputed item embeddings

Runtime: ~2 minutes on CPU.

In [ ]:
import numpy as np
rng = np.random.default_rng(42)

# ---------- Synthetic interaction data ----------
# 2000 users, 400 items, latent tastes + category affinities
n_users, n_items, n_cats, k = 2000, 400, 10, 8
P = rng.normal(0, 1, (n_users, k))          # user taste factors
Q = rng.normal(0, 1, (n_items, k))          # item latent factors
cat = rng.integers(0, n_cats, n_items)      # item category
cat_aff = rng.normal(0, 0.8, (n_users, n_cats))

scores = 1.5 * (P @ Q.T) + cat_aff[:, cat]  # true engagement scores

INTER_PER_USER = 30
inter_u, inter_i = [], []
for u in range(n_users):
    top = np.argsort(scores[u] + rng.gumbel(0, 1, n_items))[-INTER_PER_USER:]
    inter_u += [u] * INTER_PER_USER
    inter_i += list(top)
inter_u = np.array(inter_u); inter_i = np.array(inter_i)

# train on 25 of 30 interactions per user; hold out 5 for Recall@K
perm = rng.permutation(len(inter_u))[:n_users * 25]
train_u, train_i = inter_u[perm], inter_i[perm]
print(f"training pairs: {len(train_u):,} | users: {n_users} | items: {n_items}")

## The two towers

- **User tower**: user-ID embedding → MLP → 32-dim taste vector `u`
- **Item tower**: item-ID embedding + category embedding → MLP → 32-dim vector `v`
- Score = dot product `u·v`. Training loss = softmax cross-entropy over **in-batch negatives** (other positives in the same batch act as negatives — the standard cheap trick).

In [ ]:
D = 32  # embedding dim
U_emb = rng.normal(0, 0.1, (n_users, D))
E_emb = rng.normal(0, 0.1, (n_items, D))
C_emb = rng.normal(0, 0.1, (n_cats, 8))
W1 = rng.normal(0, np.sqrt(2/32), (D, 64)); b1 = np.zeros(64)
W2 = rng.normal(0, np.sqrt(2/64), (64, D)); b2 = np.zeros(D)
W3 = rng.normal(0, np.sqrt(2/40), (40, 64)); b3 = np.zeros(64)
W4 = rng.normal(0, np.sqrt(2/64), (64, D)); b4 = np.zeros(D)
params = [U_emb, E_emb, C_emb, W1, b1, W2, b2, W3, b3, W4, b4]
m = [np.zeros_like(p) for p in params]; v = [np.zeros_like(p) for p in params]
lr, b1a, b2a, eps, t = 1e-2, 0.9, 0.999, 1e-8, 0

def user_tower(uids):
    x0 = U_emb[uids]
    z1 = x0 @ W1 + b1; a1 = np.maximum(z1, 0)
    return x0, z1, a1, a1 @ W2 + b2

def item_tower(iids):
    x = np.concatenate([E_emb[iids], C_emb[cat[iids]]], axis=1)
    z3 = x @ W3 + b3; a3 = np.maximum(z3, 0)
    return x, z3, a3, a3 @ W4 + b4

def adam_step(grads):
    global t; t += 1
    for p, g, mi, vi in zip(params, grads, m, v):
        mi[:] = b1a * mi + (1 - b1a) * g
        vi[:] = b2a * vi + (1 - b2a) * g * g
        p -= lr * (mi / (1 - b1a**t)) / (np.sqrt(vi / (1 - b2a**t)) + eps)

def train_step(us, ps):
    bb = len(us)
    x0, z1, a1, u = user_tower(us)
    x, z3, a3, vv = item_tower(ps)
    S = u @ vv.T                                  # bb x bb score matrix
    Pe = np.exp(S - S.max(1, keepdims=True)); Pe /= Pe.sum(1, keepdims=True)
    loss = -np.log(Pe[np.arange(bb), np.arange(bb)] + 1e-12).mean()
    dS = (Pe - np.eye(bb)) / bb                   # in-batch sampled softmax
    du, dvv = dS @ vv, dS.T @ u
    da1 = du @ W2.T; dz1 = da1 * (z1 > 0)
    da3 = dvv @ W4.T; dz3 = da3 * (z3 > 0)
    gx = dz3 @ W3.T
    gU = np.zeros_like(U_emb);  np.add.at(gU, us, dz1 @ W1.T)
    gE = np.zeros_like(E_emb);  np.add.at(gE, ps, gx[:, :D])
    gC = np.zeros_like(C_emb);  np.add.at(gC, cat[ps], gx[:, D:])
    adam_step([gU, gE, gC, x0.T @ dz1, dz1.sum(0), a1.T @ du, du.sum(0),
               x.T @ dz3, dz3.sum(0), a3.T @ dvv, dvv.sum(0)])
    return loss

B, EPOCHS = 256, 40
for ep in range(EPOCHS):
    idx = rng.permutation(len(train_u)); tl, nb_ = 0.0, 0
    for s in range(0, len(train_u), B):
        b = idx[s:s+B]
        tl += train_step(train_u[b], train_i[b]) * len(b); nb_ += len(b)
    if (ep + 1) % 10 == 0:
        print(f"epoch {ep+1:2d}  in-batch softmax loss {tl/nb_:.4f}")
print("training done")

## Evaluation: Recall@K on held-out interactions

For each user we check whether the 5 interactions the model never trained on appear in the top-10 retrieved items. Random guessing would score ~0.025.

In [ ]:
def all_embeddings():
    _, _, _, U = user_tower(np.arange(n_users))
    _, _, _, V = item_tower(np.arange(n_items))
    return U, V

U, V = all_embeddings()
S = U @ V.T
train_set = set(zip(train_u.tolist(), train_i.tolist()))
by_user = {}
for u, i in zip(inter_u.tolist(), inter_i.tolist()):
    by_user.setdefault(u, []).append(i)

hits = tot = 0
for u, items in by_user.items():
    held = [i for i in items if (u, i) not in train_set]
    top10 = set(np.argsort(S[u])[-10:])
    hits += len(top10 & set(held)); tot += len(held)
print(f"Recall@10 = {hits/tot:.3f}  (random baseline ~0.025, i.e. ~{hits/tot/0.025:.0f}x better)")

## The retrieval demo: why two towers scale

At serving time the **item embeddings are precomputed once** and the request becomes a single user-tower pass plus a nearest-neighbor lookup. That is the whole 10-millisecond trick.

In [ ]:
# Item vectors V were computed once above — "offline". Serving = one user pass + lookup.
u0 = 0
_, _, _, u_vec = user_tower(np.array([u0]))
scores0 = (u_vec @ V.T)[0]
top5 = np.argsort(scores0)[-5:][::-1]
print(f"Top-5 recommendations for user {u0}:")
for i in top5:
    print(f"  item {i:3d}  category {cat[i]}  score {scores0[i]:+.2f}")
print("\nIn production, V lives in an ANN index (HNSW) so this lookup is O(log N), not O(N).")

## What this notebook leaves out (and the post covers)

- **Hard negatives.** We used in-batch negatives — cheap but easy. Meta's July 2026 paper (*Real-Time Hard Negative Sampling via LLM-based Clustering*, arXiv 2607.00448) samples negatives from the *same item cluster* as the positive and got **+53% CTR** in a production deployment. Try it here: replace random in-batch negatives with same-category negatives and watch Recall@K move.
- **Bias correction.** Popular items dominate in-batch negatives; production systems apply LogQ correction (Google, ByteDance, Kuaishou).
- **ANN serving.** We did brute-force dot products; real systems index `V` with HNSW/IVF for sub-10ms retrieval over millions of items.
- **Ranking stage.** Retrieval only finds 500 candidates; a heavier model (increasingly an LLM reranker, e.g. LlamaRec / LLM-RS) picks the final 10.